# 3.5 分數為什麼需要縮放？


兩特徵的點積容易看清楚，模型卻可能用六十四個或更多特徵。每格相乘後再全部相加，即使每個數都差不多大，總分的起伏也可能隨格數增加。若有一些分數明顯高過其他分數，softmax會把較多讀取比例分配給這些高分候選；但不一定只有一兩個位置，多個最高分並列時，它們可以分到相同的比例。較低分位置的比例可能接近0，卻仍要乘上讀到的內容，才能知道實際加權數值的大小，見 [3.1](https://birdhackor.github.io/tiny-perceptron-vlm/3.1.html)。我們希望增加特徵時，不要同時無意間把讀法推得特別尖銳；這裡談的是讀取哪些資料及各自占多少，不是在計算訓練時怎樣調整參數。

回看 [3.2](https://birdhackor.github.io/tiny-perceptron-vlm/3.2.html)，點積把對應特徵乘起來，再加總。我們先把每格的乘積簡化成一半機會取-1、一半機會取+1，平均便是0。量起伏時，先看每個數偏離平均多少，平方後再取平均：這裡是 `(1+1)/2=1`，叫變異數（variance）。再取變異數的平方根，得到標準差（standard deviation）；平方根是平方後會得到原數、而且不小於0的那個數，所以這裡標準差也是1。它讓我們用和原數相同的尺度讀出典型起伏。

現在加兩格。假設各格獨立，也就是知道第一格取了哪個值，不會改變第二格取-1或+1的機會。四種組合 `(-1,-1)、(-1,+1)、(+1,-1)、(+1,+1)` 各有一樣的機會，加總分別為 `-2、0、0、2`，平均仍是0。偏離量平方後取平均，得到 `(4+0+0+4)/4=2`；標準差是√2，約1.414。兩格各有變異數1，加起來的變異數為2，但標準差並沒有直接變成2，因為正負起伏也有互相抵消的組合。

對更多獨立的格子，也使用同一條變異數相加規則。若各格平均為0、變異數為1，D格加總的變異數為D，標準差就是√D。四格對應變異數4、標準差2；六十四格對應變異數64、標準差8，因此要分別除以2與8才把典型起伏拉回1。這是指定假設下的尺度推算，下面再用隨機數核對；真正訓練過的向量未必獨立、平均為0或每格變異數恰好為1，不能把這個推算當成所有模型都成立的數值保證。

因而常把匹配分數除以 `sqrt(D)`，稱縮放點積注意力（scaled dot-product attention）。這裡沿用 [3.4](https://birdhackor.github.io/tiny-perceptron-vlm/3.4.html) 的query/key：用來查詢與比對的兩排數字；D是它們各自的匹配特徵數。[多頭注意力](https://birdhackor.github.io/tiny-perceptron-vlm/3.7.html)同時用幾組各自的配方做匹配和讀取，每組叫一個head（頭）；若已分成多頭，D要數該組query/key的特徵，而非整層總寬度。下面不做完整注意力，只測分數起伏。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import math
import torch

torch.manual_seed(42)
for d in [4, 64]:
    q = torch.randn(1000, d)
    k = torch.randn(1000, d)
    scores = (q * k).sum(dim=-1)
    scaled = scores / math.sqrt(d)
    print(d, scores.std().item(), scaled.std().item())

輸入各是1000組隨機q、k，每組d個數。`randn` 產生平均約0、標準差約1的數字；固定種子便於重現。`q*k` 是逐格相乘，不是矩陣乘法，再沿最後特徵軸相加，形成1000個匹配分數。`std()` 計算這1000個分數的標準差。d=4時未縮放值應靠近2，d=64時靠近8；除以2或8後，兩者都應靠近1，具體值因取樣而有些偏差。

這個實驗的結論是「在指定隨機假設下，縮放能讓尺度較可比」，不是「所有已訓練q/k的標準差恰好1」。如果兩組向量本身數值變大，除以√D並不會自動消除全部變化。模型還需要後面章節的數值尺度控制，而且softmax是否合適集中也跟任務有關。

尺度還與 [1.15](https://birdhackor.github.io/tiny-perceptron-vlm/1.15.html) 的溫度有相似數學效果：除以大數會使分配較平，但這裡的√D是注意力匹配的固定規則，目的在控制隨特徵數增長的波動，不是生成答案時為多樣性選的溫度。

練習只把q與k各乘2，再算分數，先預測每個乘積會變四倍，所以未縮放與縮放的標準差都約四倍。執行核對；不能說縮放後一定又回1。這個反例讓你看見√D控制的是特徵數量帶來的變化，而不是所有來源的數值大小。
